# 01 — Exploration du dataset Chest X-Ray

Avant d'entraîner quoi que ce soit, on doit **voir et comprendre les données**.

Questions auxquelles on va répondre :
- À quoi ressemblent les images ?
- Quelle différence visuelle entre NORMAL et PNEUMONIA ?
- Les classes sont-elles équilibrées ?
- Quelles sont les caractéristiques des images (taille, couleur) ?

## 1. Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image
import random

# Chemin vers le dataset
DATA_PATH = Path('../data/raw/chest_xray')

print('Dataset trouvé :', DATA_PATH.exists())
print('Dossiers :', [d.name for d in DATA_PATH.iterdir() if d.is_dir()])

## 2. Compter les images par classe

On vérifie combien d'images on a et si les classes sont équilibrées.

In [ ]:
splits = ['train', 'val', 'test']
classes = ['NORMAL', 'PNEUMONIA']

counts = {}
for split in splits:
    counts[split] = {}
    for cls in classes:
        folder = DATA_PATH / split / cls
        n = len(list(folder.glob('*.jpeg'))) + len(list(folder.glob('*.jpg')))
        counts[split][cls] = n

print(f"{'Split':<10} {'NORMAL':>10} {'PNEUMONIA':>12} {'TOTAL':>8}")
print('-' * 45)
for split in splits:
    n = counts[split]['NORMAL']
    p = counts[split]['PNEUMONIA']
    print(f"{split:<10} {n:>10} {p:>12} {n+p:>8}")

In [ ]:
# Visualiser le déséquilibre dans le train
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

colors = ['steelblue', 'coral']
for i, split in enumerate(splits):
    vals = [counts[split]['NORMAL'], counts[split]['PNEUMONIA']]
    axes[i].bar(classes, vals, color=colors)
    axes[i].set_title(f'{split.upper()}', fontsize=13)
    axes[i].set_ylabel("Nombre d'images")
    for j, v in enumerate(vals):
        axes[i].text(j, v + 20, str(v), ha='center', fontweight='bold')

plt.suptitle('Distribution des classes par split', fontsize=15)
plt.tight_layout()
plt.savefig('../results/figures/class_distribution.png', dpi=150)
plt.show()

## 3. Visualiser des exemples d'images

On regarde des radiographies NORMAL vs PNEUMONIA pour comprendre visuellement ce que le modèle devra apprendre.

In [ ]:
def get_random_images(split, cls, n=4):
    folder = DATA_PATH / split / cls
    images = list(folder.glob('*.jpeg')) + list(folder.glob('*.jpg'))
    return random.sample(images, min(n, len(images)))

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for col, img_path in enumerate(get_random_images('train', 'NORMAL', 4)):
    img = Image.open(img_path).convert('L')  # Niveau de gris
    axes[0, col].imshow(img, cmap='gray')
    axes[0, col].set_title('NORMAL', color='steelblue', fontsize=12)
    axes[0, col].axis('off')

for col, img_path in enumerate(get_random_images('train', 'PNEUMONIA', 4)):
    img = Image.open(img_path).convert('L')
    axes[1, col].imshow(img, cmap='gray')
    axes[1, col].set_title('PNEUMONIA', color='coral', fontsize=12)
    axes[1, col].axis('off')

plt.suptitle('Exemples de radiographies thoraciques', fontsize=15)
plt.tight_layout()
plt.savefig('../results/figures/sample_images.png', dpi=150)
plt.show()

## 4. Analyser les tailles d'images

Les réseaux de neurones ont besoin d'images de **taille fixe**. On vérifie si toutes les images ont la même taille.

In [ ]:
widths, heights = [], []

# On analyse un échantillon de 200 images du train
all_train_images = (list((DATA_PATH / 'train' / 'NORMAL').glob('*.jpeg')) +
                    list((DATA_PATH / 'train' / 'PNEUMONIA').glob('*.jpeg')))

sample = random.sample(all_train_images, min(200, len(all_train_images)))

for img_path in sample:
    with Image.open(img_path) as img:
        w, h = img.size
        widths.append(w)
        heights.append(h)

print(f'Largeur  — min: {min(widths)}px  max: {max(widths)}px  moyenne: {int(np.mean(widths))}px')
print(f'Hauteur  — min: {min(heights)}px  max: {max(heights)}px  moyenne: {int(np.mean(heights))}px')
print(f'\n→ Les images ne sont pas toutes de la même taille.')
print(f'→ On les redimensionnera toutes à 224x224 px pour le modèle (standard ResNet).')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(widths, bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('Distribution des largeurs')
axes[0].set_xlabel('Pixels')
axes[0].axvline(224, color='red', linestyle='--', label='Cible : 224px')
axes[0].legend()

axes[1].hist(heights, bins=30, color='coral', edgecolor='white')
axes[1].set_title('Distribution des hauteurs')
axes[1].set_xlabel('Pixels')
axes[1].axvline(224, color='red', linestyle='--', label='Cible : 224px')
axes[1].legend()

plt.suptitle('Tailles des images (échantillon de 200)', fontsize=13)
plt.tight_layout()
plt.savefig('../results/figures/image_sizes.png', dpi=150)
plt.show()

## 5. Conclusions

Note ici ce que tu observes :

**Déséquilibre des classes :**
- Il y a environ 3x plus d'images PNEUMONIA que NORMAL dans le train
- Comme pour le projet HAR, on devra gérer ça (class weights ou F1-score)

**Différences visuelles :**
- Poumons NORMAUX : zones pulmonaires claires et bien définies
- PNEUMONIA : zones blanches/floues dans les poumons (opacités = infection)

**Taille des images :**
- Les images ont des tailles variables → on les redimensionnera à **224×224 px**
- C'est la taille standard pour ResNet (le modèle qu'on utilisera)

---
**Prochain notebook :** `02_preprocessing.ipynb` — normalisation et data augmentation.